### Etapa 1: Buscador de Imágenes por Similitud
El sistema deberá:

● Extraer embeddings de imágenes.

● Construir una base de datos vectorial.

● Recuperar imágenes similares.

● Clasificar razas utilizando búsqueda por similitud.


In [1]:
import os
import sys
import uuid
from typing import Any, Dict, List, Optional, Tuple
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
from tqdm.auto import tqdm
from PIL import Image
from dotenv import load_dotenv


load_dotenv(dotenv_path=".env.docker.example")

import sys
sys.path.append(os.path.abspath('./src'))
from lib.storage.pgvector_store import PgVectorEmbeddingStore
from lib.schemas import EmbeddingRecord

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

c:\Users\Martin\Desktop\Computer Vision\tuia-dog-recognition-app\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Usando dispositivo: cpu


In [2]:
store = PgVectorEmbeddingStore(
    dbname=os.getenv("POSTGRES_DB","dogs"),
    user=os.getenv("POSTGRES_USER", "dogs_user"),
    password=os.getenv("POSTGRES_PASSWORD", "dogs_pass"),
    host=os.getenv("POSTGRES_ALT_HOST", "localhost"),
    port=int(os.getenv("POSTGRES_PORT", "5432")),
    embedding_dim=int(os.getenv("EMBEDDING_DIM", 1280))
)

store.truncate()

In [3]:
DATASET_PATH = os.getenv("DATASET_PATH")
DATA_PATH = os.getenv("DATA_PATH")
IMAGE_SIZE = os.getenv("IMAGE_SIZE")
EMBEDDING_DIMG = os.getenv("EMBEDDING_DIM")
TOP_K = os.getenv("TOP_K")

In [4]:
weights = models.EfficientNet_B0_Weights.DEFAULT
base_model = models.efficientnet_b0(weights=weights)

class EfficientNetEmbedding(nn.Module):
    """
    Modelo personalizado para extraer embeddings de EfficientNet-B0.
     - Se eliminan las capas de clasificación y se mantiene la parte de extracción de características.
     - El método forward devuelve un vector de características de tamaño 1280.
    """
    def __init__(self, base_model):
        super().__init__()
        self.features = base_model.features
        self.pool = base_model.avgpool

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return torch.flatten(x, 1)

model = EfficientNetEmbedding(base_model)
model.eval() 

EfficientNetEmbedding(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNo

In [5]:
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [6]:
batch_size = 32

train_dir = f'{DATASET_PATH}/train'
train_dataset = datasets.ImageFolder(root=train_dir, transform=preprocess)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

test_dir = f'{DATASET_PATH}/test'
test_dataset = datasets.ImageFolder(root=test_dir, transform=preprocess)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

In [7]:
def extraer_embeddings(loader: DataLoader, split_name: str, model: torch.nn.Module, device: str, persistencia: bool = False, store: Optional[PgVectorEmbeddingStore] = None) -> Tuple[int, List[Dict[str, Any]]]:
    """
    Extrae embeddings de un DataLoader, los normaliza y opcionalmente los persiste en pgvector.
    
    Argumentos:
        loader: El DataLoader con los datos a procesar.
        split_name (str): El nombre del dataset.
        model: El modelo de PyTorch a utilizr.
        device: 'cuda' o 'cpu'.
        persistencia (bool): Si es True, guarda en la base de datos.
        store: La instancia de PgVectorEmbeddingStore (requerido si persistencia=True).
    
    Funcionamiento:
        - Itera sobre los batches del DataLoader.
        - Para cada batch, mueve las imágenes al dispositivo, extrae los embeddings, los normaliza y los convierte a numpy.
        - Para cada imagen en el batch, obtiene la ruta completa, la clase, genera un ID único y crea un registro.
        - Si persistencia es True, guarda el registro en la base de datos usando store.append_batch().
            - Este proceso se hace en batches para evitar crear una conexión nueva a la base de datos cada vez que persistimos un registro. (Ahorro de ~7 min en la extracción total).
        - Si persistencia es False, acumula los registros en una lista para su posterior uso.
    """
    idx_global = 0  
    registros_no_persistidos = []
    dataset = loader.dataset 
    print(f"Iniciando extracción para el set de '{split_name}'...")
    model.eval()

    with torch.no_grad():
        for images, labels in tqdm(loader, desc=f"Procesando batches ({split_name})"):
            
            images = images.to(device)
            
            embeddings = model(images)
            embeddings_norm = F.normalize(embeddings, p=2, dim=1)
            embeddings_np = embeddings_norm.cpu().numpy()
            batch_registros = []
            for i in range(len(images)):

                ruta_completa, class_idx = dataset.samples[idx_global]
                ruta_normalizada = ruta_completa.replace('\\', '/')
                breed = dataset.classes[class_idx]
                id_unico = str(uuid.uuid4())
                embedding_lista = embeddings_np[i].tolist()
                if persistencia and store is not None:
                    record = EmbeddingRecord(
                        id_imagen=id_unico,
                        path=ruta_normalizada,    
                        breed=breed,           
                        embedding=embedding_lista, 
                        metadata={"split": split_name}
                    )
                    batch_registros.append(record)
                else:
                    registro = {
                        "id_imagen": id_unico,
                        "ruta": ruta_normalizada,
                        "breed": breed,          
                        "embedding": embedding_lista, 
                        "metadata": {"split": split_name}
                    }
                    registros_no_persistidos.append(registro)

                idx_global += 1

            
            if persistencia and store is not None and batch_registros:
                store.append_batch(batch_registros)

    print(f"Extracción para '{split_name}' completada.")
    return idx_global, registros_no_persistidos

In [8]:
cont_train, _ = extraer_embeddings(train_loader, "train", model, device, persistencia=True, store=store)

Iniciando extracción para el set de 'train'...


Procesando batches (train): 100%|██████████| 249/249 [03:51<00:00,  1.07it/s]


Extracción para 'train' completada.


In [9]:
cont_test, embeddings_test = extraer_embeddings(test_loader, "test", model, device, persistencia=False, store=None)

Iniciando extracción para el set de 'test'...


Procesando batches (test):   0%|          | 0/22 [00:00<?, ?it/s]

Procesando batches (test): 100%|██████████| 22/22 [00:21<00:00,  1.02it/s]


Extracción para 'test' completada.


In [10]:
print("Imágenes de Train procesadas:", cont_train)
print("Imágenes de Test procesadas:", cont_test)

Imágenes de Train procesadas: 7946
Imágenes de Test procesadas: 700


In [11]:
print(f"Embedding de Test 0: {embeddings_test[0]}")

Embedding de Test 0: {'id_imagen': '5aaeb650-3bf8-4174-a478-36139a433fdb', 'ruta': 'data/dataset/test/Afghan/01.jpg', 'breed': 'Afghan', 'embedding': [0.03687790036201477, -0.002560366177931428, -0.010271686129271984, 0.0036311913281679153, -0.016716301441192627, -0.013669641688466072, 0.019545409828424454, -0.0076060607098042965, -0.004555596504360437, 0.019221292808651924, -0.015007690526545048, 0.02097095362842083, -0.0007580473320558667, -0.014559929259121418, -0.01320138480514288, -0.014422588050365448, -0.015296024270355701, -0.0030586691573262215, -0.015211780555546284, -0.018295185640454292, -0.007239929866045713, 0.01760280877351761, -0.006260280963033438, -0.013588194735348225, -0.013197293505072594, 0.10493236035108566, -0.008398275822401047, 0.0415172316133976, -0.015064455568790436, -0.013130868785083294, -0.01646549068391323, 0.06476958841085434, -0.012271273881196976, -0.01401426363736391, 0.0018316206987947226, -0.012510702013969421, -0.009771367534995079, -0.0136078475